# ML

### Grouped, stratified train/test split
All experiments share one split. Complete drillholes stay together; stratification aims for similar Au-class proportions but cannot guarantee them. Of five fixed-seed folds, the one closest to 20% of rows becomes the test set. Selection uses sample count only, never model performance.

Binary measured-Au labels: Normal < 1 ppm; Anomaly >= 1 ppm (including exactly 1 ppm). Normal means below this Au screening threshold, not geologically unaltered. This is supervised classification, not unsupervised anomaly detection. Run data preparation first; preprocessing is fitted after splitting.

This is a new exploratory target chosen after inspecting the three-class results. Stratification is recomputed for the binary labels, so the held-out holes may change. Scores are not directly comparable with the earlier three-class experiment and are not independent confirmation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# constants
ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()
DATASET_PATH = ROOT / "data" / "extracted_data" / "ml_dataset.csv"
RANDOM_STATE = 67
N_SPLITS = 5
TEST_FRACTION = 0.2
ANOMALY_AU_PPM = 1.0
CLASS_NAMES = ["Normal", "Anomaly"]

In [ ]:
df = pd.read_csv(DATASET_PATH)

# Labels are measured Au classes, not predictions of the model.
is_anomaly = df["Au_PPM"] >= ANOMALY_AU_PPM
df["Au_Class"] = np.where(is_anomaly, "Anomaly", "Normal")

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

grouped_splits = list(splitter.split(df, df["Au_Class"], groups=df["CollarId"]))

test_size_differences = [
    abs(len(test_rows) / len(df) - TEST_FRACTION) for _, test_rows in grouped_splits
]
selected_fold = test_size_differences.index(min(test_size_differences))
train_indices, test_indices = grouped_splits[selected_fold]

df_train = df.iloc[train_indices].copy()
df_test = df.iloc[test_indices].copy()

train_drillholes = set(df_train["CollarId"])
test_drillholes = set(df_test["CollarId"])

assert train_drillholes.isdisjoint(test_drillholes), "Drillhole leakage"
assert len(df_train) + len(df_test) == len(df), "Incomplete split"

In [ ]:
split_overview = pd.DataFrame(
    {
        "rows": [len(df_train), len(df_test)],
        "rows_percent": [len(df_train) / len(df), len(df_test) / len(df)],
        "drillholes": [len(train_drillholes), len(test_drillholes)],
    },
    index=["Train", "Test"],
)

split_overview["rows_percent"] *= 100

class_counts = (
    pd.DataFrame(
        {
            "Train": df_train["Au_Class"].value_counts(),
            "Test": df_test["Au_Class"].value_counts(),
        }
    )
    .reindex(CLASS_NAMES)
    .fillna(0)
    .astype(int)
)

class_percentages = class_counts.div(class_counts.sum()) * 100

print("Split overview:")
print(split_overview.round(1).to_string())
print("\nAu-class counts:")
print(class_counts.to_string())
print("\nAu-class percentages within each split:")
print(class_percentages.round(1).to_string())
print("\nShared drillholes: 0")

### Feature sets for later modelling
Use these explicit columns from the same `df_train` and `df_test` for all three experiments. Metadata stays in the tables for interpretation, but is not a predictor.

In [ ]:
swir_features = [
    "900D_Fe",
    "900W_Fe",
    "1900D_Water",
    "2200D_AlOH",
    "2200W_AlOH",
    "2250D_FeOH",
    "2250W_FeOH",
]

tir_features = ["8625D_Quartz", "11300D_Carbonate", "11300W_Carbonate"]

feature_sets = {
    "SWIR": swir_features,
    "TIR": tir_features,
    "combined": swir_features + tir_features,
}

y_train = df_train["Au_Class"]
y_test = df_test["Au_Class"]

### Classification pipelines
Each feature set receives a fresh Dummy, Random Forest, and XGBoost pipeline. Median imputation is fitted on training data only. Missingness indicators are separate predictors; all-missing training columns are retained. Tree models need neither scaling nor a target log transform.

Random Forest uses balanced class weights. XGBoost receives equivalent per-sample weights computed from training labels only and optimizes binary cross-entropy (`binary:logistic`, `logloss`). These weights do not correct project imbalance or calibrate probabilities. The Dummy uses the unweighted training class prior. Model decisions use each classifier's default rule, without threshold tuning. All settings are fixed before test evaluation; there is no test-based tuning or early stopping.

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

CLASS_IDS = list(range(len(CLASS_NAMES)))
CLASS_TO_ID = dict(zip(CLASS_NAMES, CLASS_IDS, strict=True))
ANOMALY_CLASS_ID = CLASS_TO_ID["Anomaly"]
N_JOBS = 2

# Keep the screening class order consistent across both libraries.
y_train_encoded = y_train.map(CLASS_TO_ID)
y_test_encoded = y_test.map(CLASS_TO_ID)
if y_train_encoded.isna().any() or y_test_encoded.isna().any():
    raise ValueError("Unknown Au class; check CLASS_NAMES and training labels")
if set(y_train_encoded) != set(CLASS_IDS):
    raise ValueError("Training requires examples of both Au classes")

training_weights = compute_sample_weight(class_weight="balanced", y=y_train_encoded)
classifiers = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        tree_method="hist",
        importance_type="gain",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
}

In [ ]:
fitted_models = {}

for feature_set_name, feature_columns in feature_sets.items():
    X_train = df_train[feature_columns]

    for model_name, classifier in classifiers.items():
        pipeline = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median", add_indicator=True, keep_empty_features=True
                    ),
                ),
                ("classifier", clone(classifier)),
            ]
        )

        fit_parameters = {}
        if model_name == "XGBoost":
            fit_parameters["classifier__sample_weight"] = training_weights
        pipeline.fit(X_train, y_train_encoded, **fit_parameters)

        experiment = (feature_set_name, model_name)
        fitted_models[experiment] = pipeline

### Fixed holdout comparison
Evaluate each fitted pipeline on the same held-out drillholes. Anomaly precision, recall, and F1 refer specifically to Au >= 1 ppm. Macro-F1 and balanced accuracy give both classes equal weight; log loss evaluates probability scores. Per-class reports, confusion matrices, and predictions with original metadata remain available in the corresponding dictionaries for later interpretation. Do not use these test results to tune the models.

In [ ]:
prediction_tables = {}
classification_reports = {}
confusion_matrices = {}
metric_rows = []

for experiment, pipeline in fitted_models.items():
    feature_set_name, model_name = experiment
    feature_columns = feature_sets[feature_set_name]
    X_test = df_test[feature_columns]
    predicted_classes = pipeline.predict(X_test)
    class_probabilities = pipeline.predict_proba(X_test)
    assert np.array_equal(pipeline.classes_, CLASS_IDS)

    metric_rows.append(
        {
            "feature_set": feature_set_name,
            "model": model_name,
            "anomaly_precision": precision_score(
                y_test_encoded,
                predicted_classes,
                pos_label=ANOMALY_CLASS_ID,
                zero_division=0,
            ),
            "anomaly_recall": recall_score(
                y_test_encoded,
                predicted_classes,
                pos_label=ANOMALY_CLASS_ID,
                zero_division=0,
            ),
            "anomaly_f1": f1_score(
                y_test_encoded,
                predicted_classes,
                pos_label=ANOMALY_CLASS_ID,
                zero_division=0,
            ),
            "macro_f1": f1_score(
                y_test_encoded,
                predicted_classes,
                labels=CLASS_IDS,
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_test_encoded, predicted_classes
            ),
            "log_loss": log_loss(y_test_encoded, class_probabilities, labels=CLASS_IDS),
        }
    )

    classification_reports[experiment] = pd.DataFrame(
        classification_report(
            y_test_encoded,
            predicted_classes,
            labels=CLASS_IDS,
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).T
    confusion_matrices[experiment] = pd.DataFrame(
        confusion_matrix(y_test_encoded, predicted_classes, labels=CLASS_IDS),
        index=pd.Index(CLASS_NAMES, name="Measured"),
        columns=pd.Index(CLASS_NAMES, name="Predicted"),
    )

    predictions = df_test.copy()
    predictions["Predicted_Au_Class"] = np.array(CLASS_NAMES)[predicted_classes]
    probability_columns = [f"score_{label}" for label in CLASS_NAMES]
    predictions[probability_columns] = class_probabilities
    prediction_tables[experiment] = predictions

metrics = pd.DataFrame(metric_rows).set_index(["feature_set", "model"])
print("Held-out drillhole results; higher F1/accuracy and lower log loss are better:")
print(metrics.round(3).to_string())

### Built-in feature importances
Random Forest reports normalized impurity reduction; XGBoost reports normalized gain. These are different model-specific measures, not causal effects or directly comparable across models. Correlated descriptors can share or distort importance. `missingindicator_` entries describe missingness, not measured absorption.

These first summaries use the fitted models only, not test labels. Validate predictive importance with grouped validation and permutation importance later. The held-out metrics are a fixed-baseline comparison, not a tuning objective. No transfer to new projects or other commodities has been established.

In [ ]:
feature_importances = {}
for experiment, pipeline in fitted_models.items():
    feature_set_name, model_name = experiment
    if model_name == "Dummy":
        continue

    imputer = pipeline.named_steps["imputer"]
    classifier = pipeline.named_steps["classifier"]
    predictor_names = imputer.get_feature_names_out()
    importance = pd.Series(
        classifier.feature_importances_,
        index=predictor_names,
        name="importance",
    ).sort_values(ascending=False)
    feature_importances[experiment] = importance

    print(f"\n{feature_set_name} / {model_name}:")
    print(importance.round(4).to_string())

### Training-data spectral exploration
These descriptive plots use only measured values from `df_train`, never imputed values or held-out samples. The Fe-OH depth/position pair is chosen explicitly for geological exploration, not claimed to be the two highest-ranked features of every model. D describes absorption depth; W describes wavelength position, not width.

Points are intervals, not independent drillholes. Class overlap and project differences are descriptive observations, not causal evidence. No regression line is fitted to binary labels. PNG files stay in the Git-ignored extracted-data directory.

In [ ]:
import matplotlib.pyplot as plt

PLOT_DIR = ROOT / "data" / "extracted_data" / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DPI = 180
DEPTH_FEATURE = "2250D_FeOH"
POSITION_FEATURE = "2250W_FeOH"
FEATURE_AXIS_LABELS = {
    DEPTH_FEATURE: "2250D_FeOH - absorption depth",
    POSITION_FEATURE: "2250W_FeOH - wavelength position (nm)",
    "2200D_AlOH": "2200D_AlOH - absorption depth",
    "11300D_Carbonate": "11300D_Carbonate - absorption depth",
}
CLASS_COLORS = {"Normal": "#52677f", "Anomaly": "#c6532f"}
CLASS_MARKERS = {"Normal": "o", "Anomaly": "^"}
jitter_rng = np.random.default_rng(RANDOM_STATE)

In [ ]:
distribution_fig, distribution_axes = plt.subplots(
    2, 2, figsize=(11, 9), layout="constrained"
)
distribution_counts = {}

for axis, feature in zip(distribution_axes.flat, FEATURE_AXIS_LABELS, strict=True):
    observed_by_class = [
        df_train.loc[df_train["Au_Class"] == label, feature].dropna()
        for label in CLASS_NAMES
    ]
    distribution_counts[feature] = [len(values) for values in observed_by_class]
    tick_labels = [
        f"{label}\n(n={len(values)})"
        for label, values in zip(CLASS_NAMES, observed_by_class, strict=True)
    ]
    axis.boxplot(observed_by_class, tick_labels=tick_labels, showfliers=False)

    for position, (label, values) in enumerate(
        zip(CLASS_NAMES, observed_by_class, strict=True), start=1
    ):
        jitter = jitter_rng.uniform(-0.12, 0.12, size=len(values))
        axis.scatter(
            position + jitter,
            values,
            s=12,
            alpha=0.25,
            color=CLASS_COLORS[label],
            marker=CLASS_MARKERS[label],
        )

    axis.set_ylabel(FEATURE_AXIS_LABELS[feature])
    axis.grid(axis="y", alpha=0.2)

distribution_fig.suptitle("Measured spectral descriptors by Au class | training only")
distribution_fig.savefig(PLOT_DIR / "spectral_by_class.png", dpi=PLOT_DPI)
plt.show()
plt.close(distribution_fig)
print("Measured-value counts (missing values omitted):")
print(pd.DataFrame(distribution_counts, index=CLASS_NAMES).to_string())

In [ ]:
feoh_pairs = df_train.dropna(subset=[DEPTH_FEATURE, POSITION_FEATURE])
pair_fig, pair_axis = plt.subplots(figsize=(7, 5), layout="constrained")

for label in CLASS_NAMES:
    class_samples = feoh_pairs.loc[feoh_pairs["Au_Class"] == label]
    pair_axis.scatter(
        class_samples[POSITION_FEATURE],
        class_samples[DEPTH_FEATURE],
        color=CLASS_COLORS[label],
        marker=CLASS_MARKERS[label],
        alpha=0.4,
        s=22,
        label=f"{label} (n={len(class_samples)})",
    )

pair_axis.set(
    xlabel=FEATURE_AXIS_LABELS[POSITION_FEATURE],
    ylabel=FEATURE_AXIS_LABELS[DEPTH_FEATURE],
    title="Fe-OH position and depth by Au class | training only",
)
pair_axis.legend(title=f"Anomaly: Au >= {ANOMALY_AU_PPM:g} ppm")
pair_axis.grid(alpha=0.2)
pair_fig.savefig(PLOT_DIR / "feoh_position_vs_depth.png", dpi=PLOT_DPI)
plt.show()
plt.close(pair_fig)
print(f"Paired measured descriptors: {len(feoh_pairs)} / {len(df_train)} rows")

### Fe-OH depth versus Al-OH depth
Colour and marker shape indicate measured Au class: Normal < 1 ppm, Anomaly >= 1 ppm. Separate ordinary least-squares lines describe Al-OH depth as a function of Fe-OH depth within each class, over its observed X range only. They are not gold predictions or classifier boundaries. Intervals from the same hole are not independent; no inferential significance or confidence intervals are claimed. Only complete, measured training pairs are plotted.

In [ ]:
ALOH_DEPTH_FEATURE = "2200D_AlOH"
feoh_aloh_pairs = df_train.dropna(subset=[DEPTH_FEATURE, ALOH_DEPTH_FEATURE])
feoh_aloh_fig, feoh_aloh_axis = plt.subplots(figsize=(8, 5), layout="constrained")
REGRESSION_LINE_STYLES = {"Normal": "-", "Anomaly": "--"}
regression_fits = {}

for label in CLASS_NAMES:
    class_samples = feoh_aloh_pairs.loc[feoh_aloh_pairs["Au_Class"] == label]
    x_values = class_samples[DEPTH_FEATURE]
    y_values = class_samples[ALOH_DEPTH_FEATURE]
    feoh_aloh_axis.scatter(
        x_values,
        y_values,
        color=CLASS_COLORS[label],
        marker=CLASS_MARKERS[label],
        alpha=0.35,
        s=22,
        label=f"{label} (n={len(class_samples)})",
    )

    has_enough_distinct_x = x_values.nunique() >= 2
    if not has_enough_distinct_x:
        print(f"{label}: no regression line; need two distinct measured X values")
        continue

    slope, intercept = np.polyfit(x_values, y_values, deg=1)
    regression_fits[label] = {"slope": slope, "intercept": intercept}
    x_line = np.array([x_values.min(), x_values.max()])
    y_line = slope * x_line + intercept
    feoh_aloh_axis.plot(
        x_line,
        y_line,
        color=CLASS_COLORS[label],
        linestyle=REGRESSION_LINE_STYLES[label],
        linewidth=2,
        label=f"{label} linear fit",
    )

feoh_aloh_axis.set(
    xlabel=FEATURE_AXIS_LABELS[DEPTH_FEATURE],
    ylabel=FEATURE_AXIS_LABELS[ALOH_DEPTH_FEATURE],
    title="Fe-OH versus Al-OH absorption depth | training only",
)
feoh_aloh_axis.legend(title=f"Anomaly: Au >= {ANOMALY_AU_PPM:g} ppm")
feoh_aloh_axis.grid(alpha=0.2)
feoh_aloh_fig.savefig(PLOT_DIR / "feoh_vs_aloh_depth_by_class.png", dpi=PLOT_DPI)
plt.show()
plt.close(feoh_aloh_fig)

### Collar coordinates for all cohort holes
This table uses the full prepared cohort, not only training holes. Primary coordinates come from `WAMEX_Collar.csv`, joined by `CollarId`; the HyLogger index pair is retained separately for comparison. Package metadata specifies GDA2020 decimal degrees (geographic 2D CRS EPSG:7844), not WGS84 or projected eastings/northings.

Source: GSWA HyLogger package metadata, `METADATA_INTERIM_Hylogger_GDA2020.pdf`, p. 1; column definitions in `DATADICTIONARY_INTERIM_Hylogger_GDA2020.pdf`, pp. 4-5. These are collar locations, not coordinates of subsurface intervals. Differences between the two sources do not establish which is more accurate. The derived CSV stays local and Git-ignored; no coordinates are added as model predictors.

In [ ]:
CSV_DIR = (
    ROOT / "data" / "INTERIM_HyLogger_Dataset_Package_GDA2020_DB" / "DATABASES" / "CSV"
)
cohort_holes = df[["Dataset_ID", "CollarId", "Project"]].drop_duplicates()
collars = pd.read_csv(
    CSV_DIR / "WAMEX_Collar.csv",
    usecols=["CollarId", "Longitude", "Latitude"],
)
index_coordinates = pd.read_csv(
    CSV_DIR / "HyLogger_Dataset_Index.csv",
    usecols=["Data_name", "Shapefile_longitude", "Shapefile_latitude"],
).rename(columns={"Data_name": "Dataset_ID"})
# Unselected holes can have duplicate names in the package index.
index_coordinates = index_coordinates.loc[
    index_coordinates["Dataset_ID"].isin(cohort_holes["Dataset_ID"])
]

coordinates = cohort_holes.merge(
    collars, on="CollarId", how="left", validate="one_to_one"
).merge(index_coordinates, on="Dataset_ID", how="left", validate="one_to_one")
coordinate_columns = [
    "Longitude",
    "Latitude",
    "Shapefile_longitude",
    "Shapefile_latitude",
]
if not np.isfinite(coordinates[coordinate_columns]).all().all():
    raise ValueError("Missing or nonfinite collar coordinates in the cohort")

coordinates["Coordinate_CRS"] = "GDA2020 / EPSG:7844"
coordinates["Coordinate_Source"] = "WAMEX_Collar.csv"
COORDINATE_PATH = ROOT / "data" / "extracted_data" / "drillhole_coordinates.csv"
coordinates.to_csv(COORDINATE_PATH, index=False)
print(
    coordinates[["Dataset_ID", "Project", "Longitude", "Latitude"]].to_string(
        index=False, float_format="%.8f"
    )
)
print(f"\n{len(coordinates)} collars in GDA2020; wrote {COORDINATE_PATH}")